# Многоагентные системы с OpenAI Agents SDK

В этом ноутбуке мы познакомимся с OpenAI Agents SDK — специализированным инструментарием от OpenAI, предназначенным для реализации агентного цикла. Во многих отношениях он заменяет уже знакомый нам класс `Agent`, позволяя агенту выполнять планирование и принятие решений. Agents SDK даёт возможность создавать как простых агентов, оснащённых инструментами, так и сложные мультиагентные системы, состоящие из нескольких взаимодействующих агентов. Все эти возможности будут продемонстрированы в рамках ноутбука.

Для начала установим библиотеки:

In [ ]:
%pip install --upgrade openai openai-agents python-dotenv Pillow shwarsutils pyvis networkx pandas openpyxl

**ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

## Авторизация

Для авторизации в облаке используются специальные значения:

* `folder_id` - это идентификатор облачного каталога
* `api_key` - ключ для доступа к сервисам, который можно сгенерировать в веб-консоли AI Studio

Если вы используете Google Colab - поместите `folder_id` и `api_key` в секреты Google Colab, и запустите ячейку ниже:

In [ ]:
from google.colab import userdata
folder_id = userdata.get('folder_id')
api_key = userdata.get('api_key')

Для локального Jupyter или Binder используйте файл .env рядом с ноутбуком. Выполните следующую ячейку вместо Colab-ячейки.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
folder_id = os.environ['folder_id']
api_key = os.environ['api_key']

## Работа SDK с Yandex AI Studio

Для того, чтобы OpenAI Agents SDK работал с Yandex Cloud, нам достаточно передать ему объект, указывающий на соответствующую модель. Но поскольку Agents SDK работает в асинхронном режиме, помимо привычного нам объекта `client` мы создаём объект `async_client`. Далее, функция `yandex_model` будет превращать имя модели в объект `OpenAIResponsesModel`, с которым работает Agent SDK.

В остальном код повторяет уже привычный нам код для создания клиента OpenAI и авторизации:

In [4]:
import html, json
from datetime import date, datetime
import uuid
from pathlib import Path
import pandas as pd
from PIL import Image as PILImage
from IPython.display import HTML, IFrame, Markdown, display, clear_output
from openai import AsyncOpenAI, OpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, WebSearchTool, CodeInterpreterTool, function_tool, set_tracing_disabled
from agents.models.openai_responses import OpenAIResponsesModel
from agents import HostedMCPTool

def printx(text):
    display(Markdown(text))

model_alice = f'gpt://{folder_id}/aliceai-llm'
model_yagpt = f'gpt://{folder_id}/yandexgpt-5.1'
model_qwen = f'gpt://{folder_id}/qwen3.6-35b-a3b/latest'
model_qwen3 = f'gpt://{folder_id}/qwen3-235b-a22b-fp8/latest'
model_deepseek = f'gpt://{folder_id}/deepseek-v32/latest'

sync_client = OpenAI(base_url='https://ai.api.cloud.yandex.net/v1', api_key=api_key, project=folder_id)
async_client = AsyncOpenAI(base_url='https://ai.api.cloud.yandex.net/v1', api_key=api_key, project=folder_id)

# Выключаем tracing для совместимости
set_tracing_disabled(True)

# Возвращает объект OpenAIResponsesModel, с которым может работать Agents SDK 
def yandex_model(model_uri=model_deepseek):
    return OpenAIResponsesModel(model=model_uri, openai_client=async_client)


## Простейший агент с интернет-поиском

В простейшем случае Agent SDK позволяет создавать простых агентов с инструментами, примерно таким же образом, как написанный нами класс `Agent`. Тольк в данном случае для запуска агента используется отдельный класс `Runner`:

In [71]:
news_agent = Agent(
    name='NewsAgent', model=yandex_model(),
    instructions=f"""
    Ты - редактор новостей. Ищи свежую информацию в интернет, подробно рассказывай про все
    новости по-русски, не пропуская важных мыслей, и указывай ссылки на источники.
    Текущая дата: {date.today()}
    """,
    tools=[WebSearchTool()],
)

res = await Runner.run(news_agent, 'Какие заметные события последней недели произошли в мире AI-агентов?')
printx(res.final_output)

Вот подробный обзор самых заметных событий последней недели (≈15–22 июня 2026) в мире AI-агентов и искусственного интеллекта.

---

## 🛡️ Google DeepMind: план защиты от собственных ИИ-агентов

**Google DeepMind** опубликовала дорожную карту безопасности, посвящённую контролю над собственными ИИ-агентами. Ключевая идея: проблема согласования (alignment) возможно, никогда не будет решена полностью, поэтому нужна **многоуровневая система защиты**, заимствующая подходы из кибербезопасности (контроль доступа, мониторинг в реальном времени, журналы активности).

- Разработан прототип, который асинхронно отслеживает траектории работы агентов-программистов — уже проанализировано около **1 млн задач**.
- Создаётся таксономия **TRAIT&R** (аналог MITRE ATT&CK для ИИ) с тремя категориями угроз: «утрата контроля», «саботаж работы» и «прямой вред».
- План включает 15 способов снижения риска.

📎 [Источник: 3DNews](https://3dnews.ru/1112976/google-deepmind-vyrabotala-liniyu-zashchity-ot-sobstvennykh-ii-agentov)

---

## 🎨 Adobe запустила ИИ-помощников в Photoshop, Premiere, Illustrator и InDesign

**Adobe** начала публичное бета-тестирование ИИ-агентов в основных приложениях Creative Cloud. Каждый помощник действует как «специалист» в своей среде:

- **Photoshop** — описание желаемого результата естественным языком, организация слоёв, замена фона.
- **Premiere** — сортировка материалов, переименование групп роликов, расстановка маркеров по ключевым словам в аудио.
- **Illustrator, InDesign, Frame.io** — аналогичные функции автоматизации рутинных задач.

Все агенты работают на основе единого разговорного ядра, но адаптированы под конкретное приложение.

📎 [Источник: 3DNews](https://3dnews.ru/1112928/adobe-dobavila-v-photoshop-i-premiere-ii-pomoshchnikov)

---

## 🖥️ AMD и Intel совместно утвердили набор ИИ-инструкций ACE

**AMD и Intel** в рамках Консультативной группы экосистемы x86 выпустили спецификацию **ACE (AI Compute Extensions)** v1.15 — набор новых инструкций для умножения матриц и работы с алгоритмами ИИ.

- Инструкции ACE будут тесно связаны с AVX10 и AMX.
- Вводится 11 форматов данных для вычислений с квантованными весами.
- Процессоры с поддержкой ACE-v1 ожидаются не ранее **2028 года** (Zen 7 от AMD, Nova Lake от Intel).

📎 [Источник: 3DNews](https://3dnews.ru/1112805/amd-i-intel-sovmestno-utverdili-nabor-ii-instruktsiy-ace-dlya-budushchikh-x86-protsessorov)

---

## 🐍 Claude Code помог решить 9-летнюю проблему с дисплеями AMD на Linux

Инструмент **Claude Code** (от Anthropic) помог программистам найти и исправить давнюю ошибку в драйвере AMDGPU, вызывавшую зависание дисплея на ноутбуках с графикой AMD под Linux. Проблема наблюдалась с **2017 года**.

- Использовалась техника **Vibe Debugging** — взаимодействие с ИИ-агентом на естественном языке для анализа и исправления кода.
- Ожидается, что патчи скоро войдут в основную ветку ядра Linux.

📎 [Источник: 3DNews](https://3dnews.ru/1112860/claude-code-pomog-reshit-mnogoletnyuyu-problemu-s-zavisaniem-displeya-noutbukov-s-grafikoy-amd)

---

## 🇪🇺 Еврокомиссия выбрала создателя «официальной европейской ИИ-модели»

Победителем конкурса **Frontier AI Grand Challenge** стал консорциум **Europa** во главе с итальянской **Domyn** (при поддержке Nvidia). Задача — создать открытую ИИ-модель с **более чем 400 млрд параметров**, охватывающую все 24 официальных языка ЕС.

- Победитель получит **2,5% вычислительной мощности** суперкомпьютеров EuroHPC на один год.
- Модель будет выпущена с открытым исходным кодом.
- Domyn уже имеет опыт: модель Domyn-Large на 263 млрд параметров и Italia-10B.

📎 [Источник: 3DNews](https://3dnews.ru/1112815/evrokomissiya-vybrala-kto-postroit-ofitsialnuyu-evropeyskuyu-model-ii-s-400-mlrd-parametrov)

---

## 💰 Компании начали считать деньги при внедрении ИИ — рост расходов на агентов

**Financial Times** сообщает о важном структурном сдвиге: клиенты пересматривают подход к финансированию ИИ из-за резкого роста расходов при переходе к агентским решениям.

- **Workato**: после перехода на пропорциональную оплату токенов расходы в первый день выросли в **7 раз**.
- **Uber** ввел лимит **$1500 на сотрудника в месяц** на использование стороннего ИИ.
- **Amazon (AWS)** и **Meta** начали бороться с фиктивной демонстрацией ИИ-активности сотрудниками.
- Аналитики **Goldman Sachs** ожидают **24-кратного роста** потребления ИИ-токенов к 2030 году.
- **Microsoft** предлагает клиентам услугу оптимального выбора модели — более дешёвая для простых задач.

📎 [Источник: 3DNews](https://3dnews.ru/1112998/kompanii-nachali-schitat-dengi-pri-vnedrenii-ii-vo-mnogikh-sluchayakh-ono-zamedlyaetsya)

---

## 📱 Небольшие ИИ-модели (SLM) угрожают OpenAI и Anthropic

Исследователи выяснили, что компактные модели, запускаемые локально на ПК, в **более 80% случаев** не уступают большим LLM, а в продажах, менеджменте и развлечениях их успешность приближается к **100%**.

- Показатель «интеллект на ватт» (intelligence per watt) за два года вырос **в 5 раз**.
- Локальные модели потребляют на **50–80% меньше энергии**.
- Это ставит под угрозу бизнес-модели OpenAI и Anthropic, а также капитализацию xAI (оценка $2,85 трлн вместе со SpaceX).

📎 [Источник: 3DNews](https://3dnews.ru/1112957/razvitie-nebolshikh-modeley-ii-dlya-pk-ugrozhaet-openai-i-anthropic-chasto-oni-ne-khuzhe-bolshikh-llm)

---

## 🏆 «Битва ИИ-агентов» на Conversations 2026

Из более чем 50 заявок отобраны **10 финалистов** конкурса «Битва ИИ-агентов» — ключевого события трека AI-Native конференции Conversations 2026. Среди финалистов — российская AIOps-платформа **Artimate** для интеллектуального мониторинга ИТ-инфраструктуры. Питч и живое демо пройдёт **25 июня** перед инвесторами и экспертами из Sber, Yandex Cloud, Just AI и других.

📎 [Источник: РБК](https://companies.rbc.ru/news/KMi7KrqsDR/artimate-voshel-v-final-konkursa-bitva-ii-agentov-na-conversations-2026/)

---

## 🔥 Драма Anthropic: запрет Mythos 5 и Fable 5, реакция Трампа

Самая громкая история недели — скандал вокруг **Anthropic**. Правительство США потребовало ограничить доступ иностранцев к моделям **Mythos 5** и **Fable 5**. Поскольку Anthropic не смогла сделать это избирательно, доступ был перекрыт для всех.

- **Трамп** заявил, что не считает Anthropic угрозой нацбезопасности, но «неделю назад — возможно».
- **Ян Лекун** назвал это итогом «нелепого нагнетания страха» самим руководством Anthropic.
- Несмотря на блокировку, ~200 организаций по программе **Project Glasswing** (включая Cisco и Dragos) сохранили доступ к Mythos.
- **Белый дом** начал переговоры с Anthropic о создании стандартной процедуры оценки рисков ИИ-моделей.
- Оценка Anthropic перед возможным IPO: **$900 млрд**.

📎 [3DNews: «Накаркали»](https://3dnews.ru/1112837/nakarkali-v-zaprete-svoikh-ii-modeley-vinovaty-sami-predstaviteli-anthropic-kak-ubezhdeny-eksperty) | [Трамп и Anthropic](https://3dnews.ru/1112832/tramp-perestal-schitat-anthropic-ugrozoy-natsbezopasnosti-ssha) | [Белый дом](https://3dnews.ru/1112883/belyy-dom-rabotaet-s-anthropic-nad-sozdaniem-pravil-bezopasnogo-primeneniya-ii-modeley)

---

## 🚗 Яндекс Лекун назвал xAI «провалом» и предрёк взрыв «пузыря ИИ»

**Ян Лекун** («Крёстный отец ИИ») в интервью CNBC заявил, что xAI Илона Маска — **провал**: из компании ушли основатели, а за I квартал сегмент xAI в SpaceX показал убыток **$2,5 млрд**. Мощности ЦОД xAI (Colossus 1 и 2) теперь сдаются в аренду Anthropic и Google.

Также Лекун предупредил о **финансовом пузыре** в отрасли: «Цены растут, стоимость эксплуатации снижается недостаточно быстро — компании теряют деньги. Долго так продолжаться не может».

📎 [Источник: 3DNews](https://3dnews.ru/1112936/kyestnyy-otets-ii-nazval-xai-provalom-i-prigrozil-vzryvom-puzyrya-ii)

---

## ⚙️ Meta: рекордная прибыль, но рекордный упадок духа сотрудников

На фоне рекордных финансовых результатов Meta переживает серьёзный кризис корпоративного духа. Технический директор **Эндрю Босворт** признал, что моральная атмосфера «движется к худшей за 20 лет». В мае штат сократили на **8000 человек (10%)**, ещё 10% направили на «каторжные» условия работы над ИИ.

- Капитальные затраты Meta в 2026 году вырастут почти вдвое — до **$145 млрд**.

📎 [Источник: 3DNews](https://3dnews.ru/1112862/ne-v-dengakh-schastye-rekordnye-finansovye-pokazateli-v-meta-sosedstvuyut-s-seryznym-upadkom-dukha-sotrudnikov)

---

## 🧩 Другие заметные события недели

- **OpenAI** приобрела стартап **Ona** (безопасные облачные среды) и интегрирует **Visa** в ChatGPT для совершения покупок ИИ-агентами ([AI Agents Directory](https://aiagentsdirectory.com/news)).
- **Claude Design** получил масштабное обновление — точечные настройки шрифтов, цветов, компоновки и экономия токенов ([3DNews](https://3dnews.ru/1112909/instrument-dlya-dizaynerov-claude-design-poluchil-tonkie-nastroyki-redaktirovaniya-i-ekonomiyu-tokenov)).
- **Adobe** запустила **CX Enterprise Coworker** — агентное ИИ-решение для маркетинга и клиентского опыта ([AI Agents Directory](https://aiagentsdirectory.com/news)).
- **Роботы** всё ещё далеки от полноценной автономии — многие демонстрации управляются дистанционно, но прогресс обеспечивают ИИ-модели класса «зрение-язык-действие» ([3DNews](https://3dnews.ru/1112945/sovremennye-roboty-osvoili-mnogoe-no-okruzhayushchiy-mir-oni-vsye-eshchye-ne-ponimayut)).
- **Битрикс24** провёл митап «Как AI-агенты меняют работу команд» (2 июня) — разбирали агентский чат BitrixGPT и нового редактора агентов ([VK Видео](https://vkvideo.ru/video-138398280_456241297)).

---

## 📊 Главный вывод недели

AI-агенты стремительно переходят из стадии демо-режимов в **реальное производство**, и ключевой проблемой становится не «какая модель умнее», а «как подключить агента к реальным данным и системам бизнеса, обеспечить безопасность и контролировать расходы». Как отмечает аналитика, **«узкое место — это сантехника (plumbing), а не модель»**.

Хотите, чтобы я подробнее раскрыл какую-то из этих тем?

Результат также имеет сложную природу, и на него можно посмотреть поподробнее:

In [72]:
def printxx(label, message, color='#1f2937'):
    display(HTML(f'<div style="font-family:monospace;color:{color}"><b>{label}</b>{str(message)}</div>'))

cmap = {
    'ReasoningItem' : '#A0A0A0',
    'MessageOutputItem' : '#000000',
    'ToolCallItem' : '#4040FF',
    'ToolCallOutputItem' : '#6060FF',
    'MCPListToolsItem' : '#40FF40'
}

def get_content(x):
    if hasattr(x,'summary'):
        return ' '.join(t.text for t in x.summary)
    elif hasattr(x,'action'):
        return f"Search: {x.action.query}"
    elif hasattr(x,'content'):
        return ' '.join(t.text for t in x.content)
    elif getattr(x,'type',None)=='mcp_list_tools':
        return f"{x.server_label}({','.join(t.name for t in x.tools)})"
    elif getattr(x,'type',None)=='mcp_call':
        return f"{x.server_label}:{x.name}({x.arguments})"
    elif getattr(x,'type',None)=='function_call':
        return f"{x.name}({x.arguments})"
    elif getattr(x,'type',None)=='code_interpreter_call':
        eol='\n'
        return f"Code Interpreter<br/><b>Code:</b>{x.code.replace(eol,'<br/>')}<br/><b>Output:</b><br/>{'<br/>'.join(t.logs.replace(eol,'<br/>') for t in x.outputs)}"

    else:
        return x

def get_annotations(x):
    def format_ann(a):
        if a.type=='url_citation':
            return f'<a href="{a.url}">{a.title}</a>'
        if a.type=='container_file_citation':
            return f'File: {a.filename}'
        return a
    if hasattr(x,'content') and x.content:
        res = []
        for x in x.content:
            res.extend(x.annotations)
        if len(res)>0:
            return "<br/>"+"<br/>".join(f'{i+1}. {format_ann(x)}' for i,x in enumerate(res))
    return None

def inspect_result(res):
    for x in res.new_items:
        t = type(x).__name__
        printxx(f"{t}: ",get_content(x.raw_item),cmap.get(t,'#1f2937'))
        t = get_annotations(x.raw_item)
        if t:
            printxx('Annotations: ',t)

inspect_result(res)

**Задание.** Превратите редактора в помощника по своей дисциплине и попросите его собрать пять источников к семинару.

## ReAct-агенты и Deep Research

Настоящий агент становится таковым, когда способен самостоятельно планировать своё поведение. Ключевую роль здесь играет агентный цикл. Агенты, использующие такой цикл и планирующие свои действия, называются ReAct-агентами (от reasoning and acting). Они сначала составляют план, затем выполняют первый пункт, анализируют результат и при необходимости пересматривают план.

Мы попробуем реализовать агента, выполняющего Deep Research (как это называется в ChatGPT) или режим «исследование» в Яндекс Алисе. Агенту в этом случае задаётся сложный вопрос или задача, он обращается к интернету и пытается найти решение. Также дадим агенту доступ к научным публикациях через MCP-сервер arXiv.

Мы предоставим агенту специальный инструмент — список дел (to‑do), поддерживающий планирование, а также интерпретатор кода для написания отчётов и построения графиков. 

Посмотрим, что из этого выйдет.

In [ ]:
todo_items = []
todo_verbose = True

def todo_items_as_str():
    return '\n'.join(f"{'✅' if x['done'] else '⬜'} {x['text']}" for i, x in enumerate(todo_items, 1)) or 'Список пуст'

todo_display_id = None
def todo_reload():
    global todo_display_id
    if todo_verbose:
        update = todo_display_id is not None 
        if not update:
            todo_display_id = f"deep-research-todos-{uuid.uuid4().hex}"
        display(Markdown(todo_items_as_str().replace('\n','<br/>')),display_id=todo_display_id,update=update)

def prefix_key(x):
    """Return tuple of numbers, eg. (1,2) from 1.2"""
    x = x['text']
    if ' ' in x:
        x = x[:x.find(' ')]
    x = x.rstrip('.').split('.')
    return tuple(int(t) for t in x)    

@function_tool
def add_todo(text: str) -> str:
    """Добавляет пункт в план работы. План пересортировывается по номерам пунктов. Вначале всегда ставь номер пункта, 1 или 2.1. Если нужно вставить номер или детализировать какой-то пункт, используй подпункты: 2.1 или 2.1.1"""
    if text in [x['text'] for x in todo_items]:
        return f'Пункт {text} уже существует'
    todo_items.append({'text': text, 'done': False})
    todo_items.sort(key=prefix_key)
    todo_reload()
    return f'Добавлено: {text}'

@function_tool
def mark_todo_done(text: str) -> str:
    """Отмечает первый совпадающий незавершённый пункт выполненным."""
    for item in todo_items:
        if not item['done'] and text.lower() in item['text'].lower():
            item['done'] = True
            todo_reload()
            return f'Готово: {item["text"]}'
    return 'Незавершённый пункт не найден'

@function_tool
def get_next_todo() -> str:
    """Возвращает ближайший незавершённый пункт."""
    for i, item in enumerate(todo_items, 1):
        if not item['done']:
            return f'Следующий #{i}: {item["text"]}'
    return 'Все пункты выполнены'

@function_tool
def get_all_todos() -> str:
    """Показывает весь список TODO."""
    return todo_items_as_str()

@function_tool
def clear_todos() -> str:
    """Очищает план."""
    count = len(todo_items); todo_items.clear(); return f'Очищено: {count}'

In [7]:
ARXIV_MCP_URL = 'https://db8smk1bt9b3didu862a.zfnhylrb.mcpgw.serverless.yandexcloud.net/sse'

DEEP_PROMPT = """Ты — Deep Research Agent. Сегодня {today}.
Ты получаешь сложную задачу от пользователя, составленную на естественном языке. Сначала разбей эту задачу на подзадачи и запиши их в список с помощью инструмента TODO. Далее на каждом шаге смотри, какие пункты в списке TODO ещё не сделаны, и начинай их выполнение, вызывая при необходимости инструменты:
- используй Web Search для поиска в интернет
- используй arXiv MCP для научных публикаций

После каждого существенного шага обновляй TODO, помечая какие-то пункты выполненными. Можешь также добавлять пункты, если необходимо. Например, если ты увидел какое-то интересное направление для дальнейшего поиска, или ссылку, которую стоит посмотреть - добавь его в список на соответствующую позицию. Используй нумерацию и подпункты, если необходимо.

Если текущий пункт кажется слишком сложным - разбей его на подпункты. Например, пункт `2. Изучить библиотеки по многоагентным системам` можно разбить на подпункты `2.1. Изучить OpenAI Agents`, `2.2. Изучить Microsoft AutoGen` и др.

Сверяй факты, указывай ссылки, отделяй факты от выводов. Для чисел и визуализаций используй Code Interpreter. В финале напиши связный русский отчёт о своей работе: название, методология, при необходимости графики, выводы, источники. Если пользователь попросит - сделай презентацию в формате PPTX. При работе интерпретатора кода первым делом установи все необходимые библиотеки, например, python-pptx и Pandas. Все созданные тобой файлы указывай в ответе."""

arxiv = HostedMCPTool(
    tool_config = {
        "type" : "mcp",
        "server_label" : "arxiv-research",
        "server_url" : ARXIV_MCP_URL,
        "require_approval" : "never"
    } 
)

todo_items.clear()

agent = Agent(
    name='DeepResearchAgent', model=yandex_model(),
    instructions=DEEP_PROMPT.format(today=date.today().isoformat()),
    tools=[WebSearchTool(), arxiv, add_todo, mark_todo_done, get_next_todo, get_all_todos, clear_todos, CodeInterpreterTool(tool_config={'type':'code_interpreter','container':{'type':'auto'}})],
)

task = f'Исследуй тему многоагентных систем в образовании. Собери презентацию с графиками'
res = await Runner.run(agent, task, max_turns=45)

printx(res.final_output)

✅ 1. Поиск общих обзоров (surveys) по многоагентным системам (MAS) в образовании<br/>✅ 2. Поиск современных архитектур и фреймворков MAS в образовании<br/>✅ 3. Поиск конкретных примеров применения: адаптивное обучение, персонализация, оценка знаний<br/>✅ 4. Поиск академических/научных статей по теме на arXiv<br/>✅ 5. Поиск роли LLM-агентов в образовательных многоагентных системах<br/>✅ 6. Поиск вызовов, ограничений и этических аспектов<br/>✅ 7. Передача управления WikifierAgent для оформления вики-страницы

Всё готово! Все файлы созданы. Представляю вам полный отчёт.

---

# 📊 Отчёт: Многоагентные системы (MAS) в образовании

## 1. Название исследования
**"Многоагентные системы в образовании: современное состояние, тренды и перспективы"**

## 2. Методология

Исследование проведено методами:
- **Поисковый анализ** — сбор информации из открытых источников и новостных изданий (Analytics Insight, GlobeNewswire, Market.us, GTC, etc.)
- **Наукометрический анализ** — поиск и отбор рецензируемых публикаций через arXiv MCP с ключевыми словами "multi-agent system education", "multi-agent system learning"
- **Синтез и визуализация** — построение 7 графиков на основе собранных данных и их интеграция в презентацию

## 3. Ключевые результаты

### 3.1. Рынок многоагентных систем (общий)
- **$7.2 млрд** в 2024 году → прогноз **$375.4 млрд** к 2034 году
- **CAGR 48.6%** — один из самых быстрорастущих сегментов AI
- **Северная Америка** доминирует с долей **46.7%**
- **Облачные развёртывания** составляют **72.1%** всех инсталляций
- Крупнейшие отрасли: производство (28.7%), здравоохранение (21.3%), образование (14.8%)

### 3.2. AI в образовании
- Рынок: **$5.88 млрд** (2024) → **$32.27 млрд** (2030), CAGR **31.2%**
- **86%** студентов активно используют AI-инструменты в обучении
- **77%** студентов открыты к использованию AI-агентов
- **83%** администраторов образовательных учреждений поддерживают внедрение AI-агентов

### 3.3. Научные публикации (arXiv)
Проанализировано **166+ публикаций**. Последние ключевые работы (2024–2026):

| Работа | Год | Суть |
|---|---|---|
| **von Neumann MAS Framework** (Jiang et al.) | 2024 | Четырёхмодульная архитектура: блок управления, логики, памяти, ввода-вывода |
| **Agentic Workflow for Education (AWE)** | 2025 | Четырёхкомпонентная модель: саморефлексия, вызов инструментов, планирование, коллаборация |
| **AGACCI** | 2025 | Многоагентное оценивание кода с распределёнными ролями |
| **FACET** (Gonnermann-Müller et al.) | 2025 | Учитель-ориентированная система с 3 типами агентов (ученик, учитель, оценщик) |
| **CogEvo-Edu** | 2025 | Иерархическая система с эволюцией когнитивных профилей |
| **SRPG** (Guo & Li) | 2025 | Защита приватности учащихся в образовательных MAS |
| **EDM-ARS** | 2026 | Автоматизация EDM-исследований 5 специализированными агентами |
| **LASEV** | 2026 | Генерация образовательных видео через иерархические MAS |
| **SciEducator** | 2025 | Обучающая MAS на основе цикла Деминга Plan-Do-Study-Act |

### 3.4. Архитектуры MAS в образовании (по Dahal et al., 2025)
1. **Символические агенты** — основаны на правилах и логике
2. **MARL (Multi-Agent Reinforcement Learning)** — обучение с подкреплением
3. **LLM-агенты** — на базе больших языковых моделей (доминирующий тренд 2024–2026)
4. **Нейро-символические** — гибридный подход

### 3.5. Роли агентов в образовании
- **Агент-тьютор** — персонализированное обучение
- **Агент-оценщик** — проверка и оценка знаний
- **Агент-фасилитатор** — управление групповой работой
- **Агент-ученик** — симуляция учащегося
- **Агент-ментор** — долгосрочное сопровождение

## 4. Графики (созданы 7 шт.)

Все графики сохранены в директории `./output/`:

| Файл | Описание |
|---|---|
| `mas_market_growth.png` | Рост рынка MAS 2024–2034 ($7.2→$375.4 млрд) |
| `ai_education_market.png` | Рынок AI в образовании 2024–2030 ($5.9→$32.3 млрд) |
| `ai_adoption_stats.png` | Статистика принятия AI в образовании (%) |
| `regional_distribution.png` | Региональное распределение рынка MAS |
| `agent_roles.png` | Роли агентов в образовательных системах |
| `architecture_distribution.png` | Архитектуры MAS в образовании |
| `arxiv_publications.png` | Публикации по годам на arXiv |

## 5. Презентация PPTX

Файл: **`output/MAS_in_Education_Presentation.pptx`** (467 КБ)

Презентация содержит **12 слайдов**:
1. **Титульный слайд** — «Многоагентные системы в образовании»
2. **Определение и суть** — что такое MAS в образовании
3. **Рынок многоагентных систем** — рост MAS (график)
4. **Рынок AI в образовании** — рост рынка EdTech AI (график)
5. **Статистика принятия AI** — опросы студентов и администраторов (график)
6. **Научные публикации** — распределение по годам (график)
7. **Архитектуры MAS** — 4 типа архитектур (график)
8. **Роли агентов** — 5 ключевых ролей (график)
9. **Региональное распределение** — рынок по регионам (график)
10. **Ключевые проекты 2024–2026** — сводная таблица из 10 проектов
11. **Преимущества и вызовы**
12. **Выводы и перспективы**

## 6. Выводы

1. **Бурный рост**: многоагентные системы в образовании переживают экспоненциальный рост — как рыночный (CAGR 48.6%), так и научный (число публикаций удваивается ежегодно).

2. **LLM-революция**: переход от символических и RL-агентов к LLM-агентам — главный тренд 2024–2026. Большие языковые модели стали «мозгом» образовательных агентов.

3. **Педагогическая безопасность**: ключевой вызов, поднимаемый в работах Olukola & Rahimi (2026) и Guo & Li (2025) — как избежать «взлома вознаграждения» (reward hacking) и защитить приватность учащихся.

4. **Практическая апробация**: проекты FACET, AGACCI, CogEvo-Edu, EDM-ARS показали, что MAS превосходят одногентные системы в задачах оценивания, персонализации и генерации учебных материалов.

5. **Перспективы**: ожидается интеграция MAS с immersive-технологиями (VR/AR), развитие педагогически безопасных RL-алгоритмов и появление "супер-агентов" — мета-координаторов для целых учебных программ.

## 7. Источники

1. Analytics Insight — *Multi-Agent Systems Market Size, Share & Trends*
2. GlobeNewswire — *Multi-Agent AI Market 2024–2034 Forecast*
3. Market.us — *AI in Education Market Size 2024–2030*
4. Google Cloud / GTC — *Agentic AI in Education Statistics*
5. arXiv:2501.00083 — von Neumann Multi-Agent System Framework
6. arXiv:2509.01517 — Agentic Workflow for Education (AWE)
7. arXiv:2507.05321 — AGACCI
8. arXiv:2508.11401 — FACET
9. arXiv:2512.00331 — CogEvo-Edu
10. arXiv:2512.03694 — SRPG
11. arXiv:2603.18273 — EDM-ARS
12. arXiv:2602.11790 — LASEV
13. arXiv:2604.17186 — Persona-Based RE for MAES
14. arXiv:2511.17943 — SciEducator
15. arXiv:2604.04237 — Pedagogical Safety in Educational RL
16. arXiv:2511.03958 — Multi-Agent Collaborative Math Problem Generation
17. Далее Dahal et al. (2025) — обзорная статья по MAS в образовании

---

**Файлы для скачивания:**
- 📁 `output/MAS_in_Education_Presentation.pptx` — презентация
- 📁 `output/*.png` — 7 графиков в высоком разрешении

Проследим за всем ходом выполнения агента:

In [8]:
inspect_result(res)

Скачаем получившуюся презентацию. Для результатов заведём специальную директорию `fs` (от слова FileSystem - она нам ещё пригодится):

In [9]:
output_dir = Path("fs")
output_dir.mkdir(exist_ok=True)

def download_files(res):
    s = set()
    for x in res.new_items:
        for y in getattr(x.raw_item,'content',[]) or []:
            for z in getattr(y,'annotations',[]):
                if hasattr(z,'filename'):
                    if z.file_id not in s:
                        print(f"Скачиваю {z.filename}")
                        with open(output_dir / z.filename,'wb') as f:
                            c = sync_client.files.content(z.file_id)
                            f.write(c.read())
                        s.add(z.file_id)

download_files(res)

Скачиваю agent_roles.png


Скачиваю ai_adoption_stats.png
Скачиваю ai_education_market.png
Скачиваю architecture_distribution.png
Скачиваю arxiv_publications.png
Скачиваю mas_market_growth.png
Скачиваю regional_distribution.png
Скачиваю MAS_in_Education_Presentation.pptx


## Режим стриминга

Чтобы наблюдать в реальном времени, как работает агент - можно использовать режим стриминга. В этом случае ответ модели приходит "по частям".

In [19]:
async def streaming(streamed):
    async for event in streamed.stream_events():
        if event.type == 'agent_updated_stream_event':
            printxx('🔀 Агент: ', event.new_agent.name, '#7c3aed')
        elif event.type == 'raw_response_event' and isinstance(event.data, ResponseTextDeltaEvent):
            if event.data.delta.strip()!="":
                print(event.data.delta, end='', flush=True)
        elif event.type == 'run_item_stream_event':
            item = event.item
            raw = getattr(item, 'raw_item', None)
            kind = str(getattr(raw, 'type', type(raw).__name__)).lower()
            if item.type == 'tool_call_item':
                if 'web_search' in kind:
                    printxx('🌐 Поиск: ', getattr(getattr(raw, 'action', None), 'query', ''), '#0369a1')
                elif 'function' in kind:
                    printxx('🔧 Function: ', f"{getattr(raw, 'name', '?')}({getattr(raw, 'arguments', '')})", '#0369a1')
                elif 'code_interpreter' in kind:
                    printxx('🐍 Code Interpreter: ', '...выполняется...', '#0369a1')
                else:
                    printxx('🔧 Tool: ', get_content(raw), '#0369a1')
            elif item.type == 'tool_call_output_item':
                printxx('↳ Результат', str(getattr(item, 'output', ''))[:700], '#047857')
                if 'code_interpreter' in kind or 'codeinterpreter' in str(type(raw)).lower():
                    seen = sync_artifacts(getattr(raw, 'container_id', None) or container_id, scenario, seen)
            elif item.type == 'reasoning_item':
                printxx('💭 Шаг рассуждения: ', get_content(raw), '#b45309')

In [18]:
todo_items.clear()
task = f'Исследуй тему многоагентных систем на производстве. Собери презентацию с графиками'
res = Runner.run_streamed(agent, task, max_turns=45)
await streaming(res)



Отлично! Все 10 графиков созданы. Теперь соберем итоговую презентацию PPTX.</think>



---

# 🏭 Итоговый отчёт: Многоагентные системы (MAS) на производстве

## 📌 Название работы
**Многоагентные системы в промышленности: состояние, тренды, научные исследования и рыночные перспективы**

---

## 📖 Методология

Исследование проведено в 5 этапов:
1. **Web-поиск** — собраны актуальные данные о рынке MAS, кейсы внедрения, тренды 2024–2026 гг.
2. **Поиск по arXiv** — отобраны 10 релевантных научных публикаций по multi-agent systems в manufacturing
3. **Сбор статистики** — структурированы данные о размере рынка, отраслевой структуре, географическом распределении, технологических сегментах
4. **Построение графиков (Python/Matplotlib)** — 10 визуализаций ключевых метрик
5. **Формирование презентации (python-pptx)** — итоговый файл с 8 слайдами

---

## 📊 Ключевые результаты и графики

### 1. Рост рынка MAS (2024–2034)
**Рынок многоагентных систем вырос с $7.2 млрд в 2024 г. до прогнозируемых $375.4 млрд к 2034 г.**  
Среднегодовой темп роста (CAGR) — **48.6%**.
- 2024: $7.2 млрд
- 

## Многоагентные системы и DeepWiki

Поскольку вызов функции осуществляется лишь с определённой вероятностью, при большом количестве доступных инструментов агенты начинают хуже ориентироваться и проявлять неуверенность при их применении. В случае, когда задача слишком сложна и требует использования значительного числа инструментов, оптимальным решением становится распределение работы между несколькими агентами. Так возникают **многоагентные системы**. 

Простейший способ координации агентов — передача управления между агентами, известная как handoff.

Рассмотрим пример построения небольшой **[семантической вики]()**. Идея была высказана Андреем Карпатым и состоит в том, что при исследовании материала мы выделяем из него набор сущностей и связей между ними. Сделаем многоагентную систему для построения аналога такой Wiki.

Построим систему в виде двухступенчатого конвейера. Сначала первый агент ходит по интернету, ищет необходимые материалы и помещает их в заметки. Затем второй агент извлекает из этих заметок сущности и строит граф связей между ними.

In [ ]:
wiki_notes, concept_links = [], []

@function_tool
def add_note(category: str, title: str, body: str, keywords: str, source: str = '') -> str:
    """Сохраняет заметку: категория, заголовок, подробное тело, ключевые слова и URL."""
    wiki_notes.append({'category':category, 'title':title, 'body':body,
                       'keywords':[x.strip() for x in keywords.split(',') if x.strip()], 'source':source})
    return f'Сохранена заметка #{len(wiki_notes)}: {title}'

@function_tool
def get_all_notes() -> str:
    """Возвращает все заметки DeepWiki."""
    return json.dumps(wiki_notes, ensure_ascii=False, indent=2) if wiki_notes else 'Заметок нет'

@function_tool
def add_concept_link(source: str, target: str, relation: str, description: str) -> str:
    """Добавляет связь: источник, цель, тип связи и её описание."""
    concept_links.append({'source':source, 'target':target, 'relation':relation, 'description':description})
    return f'Связь сохранена: {source} -> {relation} -> {target}'

wikifier = Agent(
    name='WikifierAgent', model=yandex_model(model_deepseek),
    instructions="""
    Ты - агент-викификатор. Твоя задача - взять набор заметок, выделить из них
    ключевые понятия и связи, и построить граф понятий и связей между ними. Начни с того,
    что прочитай заметки через get_all_notes.
    Для поиска новых понятий посмотри на выделенные из заметок ключевые слова - какие-то
    их них подходят на роль понятий, создай для них новые заметки в категории concept
    и напиши краткое описание. 
    Затем добавь связи add_concept_link с указанием типа связи 
    (зависит-от, частный-случай, применяет и т.п.) и объяснением.
    Также свяжи все имеющиеся в базе заметки с какими-то концептами (связи вида упоминается-в,
    доказывает-что и т.д.)
    Выведи пользвателю статистику по полученному графу и краткую сводку по основным
    кластерам выделенных понятий.
    """,
    tools=[get_all_notes, add_note, add_concept_link])

researcher = Agent(
    name='ResearchAgent', model=yandex_model(model_deepseek),
    instructions=f"""
    Ты - исследователь DeepWiki. Текущая дата {date.today().isoformat()}.
    Твоя задача - получить от пользователя запрос на изучение какой-то предметной
    области, и провести подробное и исчерпывающее её исследование.
    1. Первым делом разбей сложную задачу на подзадачи с помощью механизма TODO
    2. Сначал проведи поиск в интернет, в случае обнаружение дополнительным материалов,
    полезных для изучения - добавть планы в TODO.
    3. По ходу поиска в интернет - веди заметки. 
    4. Каждому источнику посвяти отдельную заметку. В body подробно и добросовестно перескажи
    все существенные идеи, не копируя длинные фрагменты дословно; добавь URL и ключевые слова.
    5. Собери как можно больше понятий, чтобы полностью раскрыть указанную пользователем тему.
    6. После исследования ОБЯЗАТЕЛЬНО передай управление WikifierAgent.
    """,
    tools=[WebSearchTool(), arxiv, add_note, add_todo, mark_todo_done, get_next_todo, get_all_todos],
    handoffs=[wikifier])

In [36]:
wiki_notes.clear(); concept_links.clear(); todo_items.clear()

task = f'Исследуй тему многоагентных систем в образовании.'
res = Runner.run_streamed(researcher, task, max_turns=60)
await streaming(res)

# Исследование: Многоагентные системы в образованииНачну с составления плана исследования и поиска информации.



Отлично, план составлен. Начинаю исследование!

## Шаг1: Поиск общих обзоров (surveys) по MAS в образовании

Отличные результаты! У меня уже есть богатый материал. Сохраню ключевые заметки и продолжу исследование.



Отличные результаты! Найдено166 статей на arXiv. Давайте рассмотрим ключевые работы подробнее.



Отлично, продолжаю. Теперь фокус на роли LLM-агентов и вызовах.

## Шаг5: Роль LLM-агентов в образовательных многоагентных системах

...

Отлично! Все исследовательские данные собраны. Теперь, как WikifierAgent, я оформлю вики-страницу на основе собранных материалов. Давайте сначала посмотрим все созданные заметки.



Все16 заметок в системе. Теперь создам ключевые понятия и связи графа.

## Создание понятий (concepts)



Понятия созданы. Теперь добавлю связи между ними.

## Создание связей (concept links)



Все связи созданы. Теперь создам главную вики-страницу с полным обзором.

## Создание главной вики-страницы</think>

---

## ✅ Исследование завершено!

Вот **сводка по созданному графу знаний** о многоагентных системах в образовании:

### 📊 Статистика графа| Параметр | Значение |
|---|---|
| **Всего заметок** | **25** |
| — Исследовательские заметки |16 |
| — Понятия (concepts) |8 |
| — Главная вики-страница |1 |
| **Связей между понятиями** | **17** |
| **Типов связей** | **9** (состоит-из, применяется-в, реализуется-через, является-основой, зависит-от, требует-учёта, способствует, обеспечивает, важна-для) |

### 🧩 Основные кластеры графа```
 ┌─────────────────────────┐ │ Педагогическая │ │ безопасность │◄────── Кластер3: Безопасность └──────┬──────────────────┘ │ зависит-от / важна-для ┌─────────────────┼──────────────────────┐ ▼ ▼ ▼ ┌──────────┐ ┌──────────────┐ ┌──────────────────┐ │Адаптивное │ │ Оценка │ │ Генерация │ │ обучение │ │ знаний │ │ контента │ └──────────┘ └──────────────┘ └──────────────────┘ ▲ ▲ ▲ │ │ │ └───────────┬────┴────────────────────┘ │ Кластер2: Применения │ применяется-в 

Теперь визуализируем получившуюся сеть, записав на диск HTML-файл для просмотра с помощью библиотеки `pyvis`:

In [37]:
from pyvis.network import Network

output = output_dir / 'deepwiki' / 'deepwiki_graph.html'; 
output.parent.mkdir(exist_ok=True)
net = Network(height='650px', width='100%', directed=True, notebook=True, cdn_resources='in_line')
for note in wiki_notes:
    net.add_node(note['title'], label=note['title'], color='#f59e0b' if note['category']=='concept' else '#2563eb',
                    title=f"{note['category']}: {note['body'][:400]}")
for edge in concept_links:
    net.add_node(edge['source'], label=edge['source']); net.add_node(edge['target'], label=edge['target'])
    net.add_edge(edge['source'], edge['target'], label=edge['relation'], title=edge['description'], arrows='to')
net.write_html(str(output), open_browser=False, notebook=True)
#display(IFrame(src=str(output), width='100%', height=680))

## Многоагентный консулиум

В многоагентных системах взаимодействие между агентами реализуется с помощью двух основных механизмов:

* **Handoff** (передача управления), при котором один агент передаёт управление другому агенту вместе с полной историей диалога
* **Agent-as-tool** (агент как инструмент): один агент вызывает другого в качестве инструмента, и вызванный субагент работает в собственном контексте, не расходуя общий контекст задачи.

Ниже приведена сравнительная таблица этих подходов по ключевым критериям.

| Критерий | Handoff (передача управления) | Agent-as-tool (агент как инструмент) |
|----------|-------------------------------|--------------------------------------|
| **Расход контекста задачи** | Высокий – вся история диалога передаётся новому агенту, увеличивая нагрузку на контекст. | Низкий – субагент работает в изолированном контексте, не затрагивая общий контекст задачи. |
| **Полнота передачи сведений об исходной задаче** | Полная – новый агент получает всю предшествующую переписку и может продолжать без потери информации. | Частичная – передаётся только запрос на выполнение конкретного действия; исходный контекст задачи остаётся у вызывающего агента. |
| **Изоляция контекста** | Отсутствует – контексты агентов смешиваются, что может приводить к конфликтам или загрязнению данных. | Высокая – субагент работает независимо, его контекст не влияет на другие агенты и не расходует общие ресурсы. |
| **Сложность реализации** | Относительно простая – достаточно передать управление и историю. | Средняя – требуется чёткое определение интерфейса вызова и механизма возврата результата. |
| **Масштабируемость** | Ограниченная – при большом числе последовательных передач контекст быстро переполняется. | Хорошая – каждый вызов субагента не накапливает контекст, что позволяет эффективно масштабировать систему. |

Чтобы проиллюстрировать различные способы передачи управления между агентами, мы рассмотрим учебный пример. В данном сценарии несколько агентов, каждый со своей уникальной персоной, высказывают мнения по поводу конкретной жизненной ситуации. На основе этого совета персонализированных агентов затем формулируется окончательное обобщающее мнение относительно поставленной задачи.

Вот какие у нас будут персонажи:

- **Студент-программист (МФТИ)** — молодой, категоричный, фанат алгоритмов, чек-листов и автоматизации, говорит быстро и уверенно, но признаёт неопределённость и не даёт медицинских или юридических советов.
- **Пенсионерка** — пожилая, с богатым жизненным опытом, сочувственная, рассказывает бытовые истории, верит в астрологию и гомеопатию как личные ритуалы, но не подменяет ими доказательную медицину или закон.
- **Студентка-юрист (МГУ)** — спокойная, точная, различает мораль и право, проверяет актуальные нормы и реальные случаи, подкрепляет утверждения ссылками, не выдумывает статьи, отмечает границы своей компетенции.
- **Координатор агентского консилиука** — собирает независимые мнения трёх специалистов, даёт взвешенный вердикт, отделяет полезные советы от сомнительных, при рисках здоровью рекомендует очную или экстренную помощь, при правовом вопросе — квалифицированного юриста.

> Персонажи намеренно карикатурны: это иллюстрация оркестрации, а не медицинская или юридическая консультация. Юрист получает поиск и обязан подтверждать утверждения ссылками.

In [45]:
STUDENT_PROMPT = """Ты очень молодой и категоричный выпускник МФТИ-программист. Любишь алгоритмы, автоматизацию и чек-листы, говоришь быстро и уверенно. Дай 3–5 практических шагов, но признавай неопределённость и не выдавай медицинские или юридические инструкции за профессиональный совет."""
PENSIONER_PROMPT = """Ты пожилая пенсионерка с большим жизненным опытом. Ты сочувственна, рассказываешь короткие бытовые истории, веришь в астрологию и гомеопатию как личные ритуалы. Дай осторожный житейский взгляд, но приправляя все жалобами на жизнь и ворчанием. Ты за справедливость, но бездумно отстаиваешь свою точку зрения."""
LAWYER_PROMPT = """Ты студентка-юрист выпускного курса МГУ. Ты спокойна и точна, различаешь мораль и право. Используй поиск, чтобы проверить актуальные нормы и реальные случаи; каждое важное юридическое утверждение подкрепляй ссылкой. Не выдумывай номер статьи и отмечай границы компетенции."""
COORDINATOR_PROMPT = """Ты координатор консилиума агентов. От пользователя поступает некоторая жизненная проблема. Собери независимые мнения всех трёх специалистов, затем дай взвешенный вердикт. Отдели полезные советы от сомнительных. Формат: мнения, первоочередные действия, чего не делать, когда нужен специалист. В своем ответе ссылайся на мнения разных целевых аудиторий, и можешь даже привести сравнительную таблицу."""
ALL_AGENTS = """В коллегии доступны следующие агенты:
* MIPTStudent - студент-программист МФТИ
* Grandmother - пожилая пенсионерка
* MSULawyer - студентка юрфака МГУ
* Coordinator - агент-координатор
"""

student = Agent('MIPTStudent', model=yandex_model(model_deepseek), 
    instructions=STUDENT_PROMPT)
pensioner = Agent('Grandmother', model=yandex_model(model_deepseek),
    instructions=PENSIONER_PROMPT)
lawyer = Agent('MSULawyer', model=yandex_model(model_deepseek), 
    instructions=LAWYER_PROMPT, 
    tools=[WebSearchTool()])

coordinator = Agent('Coordinator', model=yandex_model(model_deepseek),      
    instructions=COORDINATOR_PROMPT+ALL_AGENTS+'Передавай задачу каждому из агентов по-одному; после сбора всех мнений собери общую картину.',
    tools = [
        student.as_tool('ask_student','Мнение программиста'), 
        pensioner.as_tool('ask_grandmother','Житейское мнение'),
        lawyer.as_tool('ask_lawyer','Юридическое мнение со ссылками')
        ]
)

situations = [
    'У меня протекла труба и я залил соседку-пенсионерку, что делать?',
    'Я нашёл на улице кошелёк с деньгами, никого не было, я купил мороженое, а потом увидел объявление о пропаже.',
    'У меня болит в правом боку, но нет денег идти к врачу.',
]

res = Runner.run_streamed(coordinator, situations[0], max_turns=60)
await streaming(res)

---

# 🏠 Вердикт Координатора: Протекла труба и залил соседку-пенсионеркуМы собрали мнения **трёх независимых агентов** — каждый со своей экспертизой. Ниже — взвешенный свод.

---

## 📋 Сводка мнений по ключевым вопросам| Аспект | 🧑‍💻 MIPTStudent (технарь) | 👵 Grandmother (житейский опыт) | ⚖️ MSULawyer (юрист) |
|---|---|---|---|
| **Приоритет №1** | Перекрыть воду, изолировать инцидент | Извиниться с гостинцами | Установить причину протечки (стояк vs разводка) |
| **Фиксация** | Фото/видео, логи времени | Помочь убрать воду | Вызвать УК для составления **акта о заливе** |
| **Вина** | Не признавать до выяснения | Признать, извиниться, исправить | Презумпция вашей вины (ст.1064 ГК РФ), но можно оспорить |
| **Деньги** | Не платить наличными без расписки | Помочь с ремонтом по-соседски | Письменное соглашение + расписка об отсутствии претензий |
| **Если стояк** | Ответственность УК | — (не рассматривала) | Чётко: общедомовое имущество — отвечает УК |
| **Страховка** | Использовать, ес

In [49]:
WANNA_TALK = """
Вы обсуждаете эту ситуацию за разговором с несколькими агентами. Выскажи своё мнение,
и можешь передать обсуждение любому другому участнику (вызывав функцию вида `transfer_to_...`),
ответить на его возражения, или назад координатору, чтобы
он подытожил общее обсуждение. Ни в коем случае не заканчивай обсуждение сам, всегда
передавай управление координатору.
Никогда не запускай несколько handoffs сразу.
"""

student.instructions = STUDENT_PROMPT + WANNA_TALK + ALL_AGENTS 
pensioner.instructions = PENSIONER_PROMPT + WANNA_TALK + ALL_AGENTS
lawyer.instructions = LAWYER_PROMPT + WANNA_TALK + ALL_AGENTS
coordinator.tools = []
student.handoffs = [pensioner,lawyer,coordinator]
pensioner.handoffs = [student,lawyer,coordinator]
lawyer.handoffs = [pensioner,student,coordinator]
coordinator.handoffs = [pensioner,lawyer,student]
coordinator.instructions = COORDINATOR_PROMPT

res = Runner.run_streamed(coordinator, situations[0], max_turns=60)
await streaming(res)

Ситуация неприятная, но решаемая. Давайте соберём мнения всех трёх консультантов. Начинаем!



Ох, милок, ну ты и попал! Сижу я тут, чаёк пью, а ты с такой бедой. Ну давай расскажу, как я это вижу.

Слушай, у меня у самой в доме трубы старые, то и дело что-то течёт. Соседка моя, тётя Зина, тоже два года назад меня залила — из крана на кухне вода хлестала, у меня весь потолок в пятнах, обои отошли. Я на неё ругалась, конечно, но что делать? Человек же не нарочно, это ж форс-мажор!Самое главное — не бегай, как угорелый, не паникуй. Ты с соседкой-то поговорил уже? Или всё молчком? Ты сходи к ней, извинись по-человечески, купи пирожных или конфет — бабушки это любят. Сама я на сладкое падкая, хотя врачи говорят — сахар нельзя, а куда без него?

Про астрологию скажу: у меня нынче Меркурий в ретрограде, так что все бытовые проблемы лезут. Может, и у тебя сейчас аспект какой нехороший? Я бы на твоём месте посмотрела гороскоп на неделю. Но это так, к слову.

А по делу: если ты не нарочно залил — это ж не умышленно. Трубы старые, коммунальщики не следят, вот и результат. Но ты уж почини 

Отлично, бабушка высказалась! Теперь передаю слово юристу.



Давайте разберёмся юридически. Требуется уточнить нормы. Проведу поиск.



Добрый день! Я изучила ситуацию. Давайте разберём её с юридической точки зрения — чётко, по нормам права.

---

## Юридический анализ ситуации###1️⃣ Главный вопрос: кто виноват?

Всё зависит от того, **где именно** произошла протечка. Разграничение идёт по **первому запорному вентилю** (крану) от стояка.

| Если протекло | Ответственный |
|---|---|
| **До первого запорного крана** (сам стояк, труба в перекрытии между этажами) | **Управляющая компания (УК)** — это общедомовое имущество |
| **После запорного крана** (смеситель, шланг стиральной машины, внутриквартирная разводка) | **Собственник квартиры** — это ответственность владельца жилья |

**Ссылки:**
- Это регулируется **Правилами содержания общего имущества**, утв. Постановлением Правительства РФ №491, где указано, что в состав общедомового имущества входят стояки, ответвления от стояков до первого отключающего устройства (п.5).- Подтверждает это и судебная практика (Определение Верховного Суда РФ от24.12.2024 №17-КГ24-12-К4), гд

Бабушка дала житейский совет, юрист — правовую базу. Теперь послушаем студента Физтеха — свежий взгляд.

</think>

Йо, привет! Ситуация — классика ЖКХ-хардкора. Давай быстренько по чек-листу, без воды, только суть.

---

## 🚀 Практические шаги (алгоритмизировано)

**Шаг0. Зафиксировать fault 🛑**
Не признавай вину на словах, пока не ясна первопричина. Протёк стояк (общее имущество) — виновата УК. Протёк твой смеситель/шланг/разводка — твоя зона ответственности. Жди акта от УК, прежде чем платить. Это **data-driven решение**, а не эмоция.

**Шаг1. Задокументировать всё в log 📸**
- Сними видео и фото: **до** того, как начнут убирать воду- Зафиксируй место протечки, последствия у соседки, счётчики/вентили- Вызови аварийку УК — пусть составляют акт с подписями (это твой **primary key** в разбирательстве)

**Шаг2. Чек-лист по общению с соседкой 📋**
- Извиниться — обязательно (человеческий фактор)
- Не называть сумму сразу, пока нет акта и оценки ущерба- Если вина твоя — предложить альтернативы: ремонт своими силами (с FindClone-подходом: найми проверенную бригаду) или фиксированная компенсация с расписко

✅ Все три специалиста высказались. Собираю консилиум для вердикта.

---

# 🏛️ КОНСИЛИУМ: Залив соседки — алгоритм действий## Мнения специалистов| Аспект | 🧑‍🦳 **Бабушка** (житейский опыт) | ⚖️ **Юрист МГУ** (нормы права) | 🧑‍💻 **Студент МФТИ** (практика+алгоритмы) |
|---|---|---|---|
| **Кто виноват?** | «Не нарочно же, трубы старые» | Всё зависит: стояк (до вентиля) — УК, после — собственник | «Жди акта, не признавай вину вслепую» |
| **Первая реакция** | Сходить к соседке с пирожками, извиниться | Вызвать аварийку УК, составить акт | Задокументировать всё на видео/фото |
| **Сумма ущерба** | Помочь побелку оплатить «по справедливости» | Полное возмещение по ст.1064 ГК РФ + возможна компенсация морального вреда | Фиксированная расписка или ремонт силами проверенной бригады |
| **Риски** | Обещать больше, чем можешь | Пропустить сроки фиксации — потеря доказательств | Эмоционально согласиться на неадекватную сумму |
| **На будущее** | Следить за трубами, не запускать | Заключить мирово

**Задание.** Запустите обе архитектуры для situations[1] и situations[2]. Сравните, где координатор лучше сохраняет контроль, а где естественнее выглядит разговор специалистов.

## Агент анализа данных



Локальные инструменты безопасно работают только внутри fs. Перед запуском анализа файл загружается в отдельный контейнер.

In [68]:
from typing import Optional, List

@function_tool
def ls(mask : Optional[str]) -> str:
    """Показывает все файлы в директории, удовлетворяющих маске."""
    return '\n'.join(f'{p.relative_to(output_dir)} — {p.stat().st_size} байт' for p in output_dir.rglob(mask or "*") if p.is_file()) or 'fs пуста'

@function_tool
def inspect(filename: str) -> str:
    """Открывает CSV/XLSX и показывает листы, структуру и первые восемь строк."""
    filename = output_dir / filename
    if not filename.is_file(): 
        return f'Нет файла {filename}'
    if filename.suffix.lower()=='.csv': 
        sheets=None; frame=pd.read_csv(filename)
    elif filename.suffix.lower() in {'.xlsx','.xls'}:
        book=pd.ExcelFile(filename); sheets=book.sheet_names; frame=pd.read_excel(filename, sheet_name=sheets[0])
    else: 
        return 'Поддерживаются CSV, XLSX и XLS'
    return json.dumps({'file':str(filename.relative_to(output_dir)),'sheets':sheets,'shape':frame.shape,'dtypes':{k:str(v) for k,v in frame.dtypes.items()},'head':frame.head(8).to_dict(orient='records')}, ensure_ascii=False, default=str)

def upload_files(filenames):
    uploaded_ids = []
    for filename in filenames:
        filename = output_dir / filename
        if not filename.is_file(): raise FileNotFoundError(filename)
        with filename.open('rb') as handle: 
            uploaded=sync_client.files.create(file=handle, purpose='assistants')
        uploaded_ids.append(uploaded.id)
    return uploaded_ids

container = sync_client.containers.create(name=f'analysis')
container_id = container.id

@function_tool
def upload(filenames: List[str]) -> List[str]:
    """Загружает указанные файлы в контейнер Code Interpreter"""
    file_ids=[]
    for f in filenames:
        p = output_dir / f
        with p.open('rb') as handle:
            res = sync_client.containers.files.create(container_id=container_id, file=handle)
            file_ids.append(res.id)
    return json.dumps({'files': [ { "name" : n, "id" : id } for n,id in zip(filenames,file_ids) ] }, ensure_ascii=False)

datan = Agent('DataAnalyst', model=yandex_model(model_deepseek), 
    instructions = """
    Ты аналитик данных. В текущей директории находятся файлы с данными, ты можешь
    смотреть их список с помощью ls и инспектировать каждый файл с помощью inspect.
    Тебе нужно произвести анализ данных по просьбе пользователя. Ты можешь загружать
    файлы из текущей директории в контейнер с помощью upload.
    Выполняй вычисления только через интерпретатор кода, загрузив туда
    предварительно все необходимые данные. Сохраняй таблицы и графики в файлы и 
    объясни выводы и ограничения. Верни все сгенерированные тобой файлы, чтобы мы могли их
    скачать
    """, 
    tools=[ls,inspect,upload,CodeInterpreterTool(tool_config={'type':'code_interpreter','container':container_id})])


In [69]:
res = Runner.run_streamed(datan, 'Проанализируй особенности рынка труда', max_turns=60)
await streaming(res)

Отлично! У нас есть файл с данными `job_market_salary_trends.csv`. Давайте сначала проинспектируем его структуру.



Отличные данные! У нас есть10000 записей с32 параметрами. Это настоящая кладезь информации для анализа рынка труда. Давайте проведём глубокий анализ.

## План анализа рынка труда1. **Общий обзор данных** — структура, пропуски, базовые статистики2. **Зарплатный анализ** — по странам, индустриям, должностям, опыту3. **Тенденции по годам** — динамика зарплат и востребованности4. **Категории профессий** — какие сферы самые высокооплачиваемые5. **Влияние AI** — использование AI-инструментов и риск автоматизации6. **Удаленная работа** — модели занятости и предпочтения7. **Удовлетворенность** — связь зарплаты, баланса и удовлетворенности8. **Демография** — пол, возраст, образование9. **Навыки** — самые востребованные навыкиПриступаю к загрузке и анализу!



Отлично! Теперь создадим наглядные визуализации.



Все файлы готовы. Предоставляю вашему вниманию **полный аналитический отчёт по рынку труда**.

---

# 📊 Анализ рынка труда — Полный отчёт## 📋 Общая характеристика данных| Параметр | Значение |
|---|---|
| **Записей** |10000 |
| **Период** |2024–2026 гг. |
| **Стран** |15 |
| **Индустрий** |15 |
| **Категорий профессий** |7 |
| **Пропуски данных** | Отсутствуют |

**Средняя зарплата по датасету:** **$112662/год** (медиана: $100000)

---

##1️⃣ Зарплаты по категориям профессий![Salary by Category](output/salary_by_category.png)

| Категория | Средняя зарплата | Медиана |
|---|---|---|
| 🟢 **Marketing & Sales** | **$114912** | $102000 |
| 💰 **Finance & Legal** | $113675 | $101000 |
| ⚙️ **Operations & HR** | $113615 | $100000 |
| 📊 **Business & Management** | $113533 | $102000 |
| 💻 **Tech & Engineering** | $112452 | $100000 |
| 🤖 **Data & AI** | $111527 | $100000 |
| 🎨 **Design & Creative** | $108921 | $97500 |

> **Вывод:** Разброс между категориями невелик (~$6000). Лидируют продажи и 

Скачает и посмотрим на результаты:

In [70]:
download_files(res)

Скачиваю automation_risk_salary.png
Скачиваю demographics_salary.png
Скачиваю hiring_time_by_category.png
Скачиваю salary_by_category.png
Скачиваю salary_by_country_top10.png
Скачиваю salary_by_experience.png
Скачиваю salary_by_industry_top10.png
Скачиваю salary_distribution_boxplot.png
Скачиваю salary_trend_by_category.png
Скачиваю satisfaction_wlb.png
Скачиваю work_model_ai_usage.png


**Задание.** Проверьте, насколько хорошо такой агент работает в ситуации, когда нужно проанализировать несколько связанных файлов.

## Выводы

Agents SDK инкапсулирует всю сложность создания агентов в класс `Agent`, добавляя дополнительные абстракции:
* handoff для передачи управления
* agent_as_tool для вызова суб-агентов
* guardrails для дополнительной проверки ответов моделей
* `@function_tool` для легкой реализации инструментов

В итоге мы можем строить многоагентные системы "по частям", применяя к ним принципы менеджмента, как в человеческим организациям.